# 11002BC Hybrid V3.2: All-Fault Landing Progression Memory, ver05

**Purpose:** test landing-indexed graded memory for every usable fault code, while retaining the detailed V3.1 family features for Fault 204, 205, 206, and ANY_TR.

Matched exploratory comparison:

- **V3:** calendar-time fault history only
- **V3.1:** V3 + detailed landing memory for 204, 205, 206, and ANY_TR
- **V3.2-500:** V3.1 + compact all-code memory, top 500 features
- **V3.2-1000:** same candidates, top 1,000 features

Every individual code receives four focal features: previous-landing presence, last-five count, weighted last-five memory, and landings since last occurrence. Cross-side weighted memory and absolute weighted-memory asymmetry are added for every code. The current landing bin is excluded from its own feature row.

This remains an exploratory benchmark because feature ranking uses all labels. Promotion requires fold-local selection or a locked out-of-time holdout.


## Configuration

Resolves the most recent Part B run under `11002BC`'s output tree (same
resume convention as the notebook's own CELL 01 fix -- never mints a new
run directory, since this builds on an already-completed Part B pass).

In [1]:
# =============================================================================
# CONFIG
# =============================================================================
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ProcessPoolExecutor, as_completed
import json
import os
import re
import time
import traceback

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

WORK_ROOT = Path("/raid3/e296408/All_ECFRs_working")
ROW_V2_ROOT = WORK_ROOT / "ECFR_11001_ROW_V2" / "records"
BC_ROOT = WORK_ROOT / "ECFR_11002BC_ALL_ENGINE_REVISED_OPTIMIZED"
ENGINE_ROOT = BC_ROOT / "engines"

# Resume convention: reuse the most recently modified existing Part B run,
# same as the notebook's own CELL 01 fix -- never mints a new run directory,
# since this script is meant to build on an already-completed Part B pass.
_data_richness_root = BC_ROOT / "data_richness"
_existing_runs = (
    sorted(_data_richness_root.glob("run_*"), key=lambda p: p.stat().st_mtime)
    if _data_richness_root.is_dir() else []
)
if not _existing_runs:
    raise RuntimeError(
        f"No existing Part B run found under {_data_richness_root} -- "
        f"run 11002BC through CELL 04 (feature build) first."
    )
PART_B_RUN_ROOT = _existing_runs[-1]
DATA_ROOT = PART_B_RUN_ROOT / "data"
MODEL_INPUT_FILE = DATA_ROOT / "landing_level_model_inputs.parquet"
if not MODEL_INPUT_FILE.is_file():
    raise FileNotFoundError(MODEL_INPUT_FILE)

V1_OUTPUT_ROOT = PART_B_RUN_ROOT / "hybrid_trailing_history_benchmark"
OUTPUT_ROOT = PART_B_RUN_ROOT / "hybrid_v3_2_all_fault_landing_memory_ver05"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
# Reuse the already-computed V1 trailing cache; do not recompute completed work.
TRAILING_CACHE_ROOT = V1_OUTPUT_ROOT / "trailing_features_by_engine"
FAULT_CACHE_ROOT = PART_B_RUN_ROOT / "hybrid_v3_fault_history_benchmark" / "fault_history_by_engine"
FAULT_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
CANONICAL_FAULT_FILE = DATA_ROOT / "canonical_fault_events.parquet"
LANDING_MEMORY_CONTRACT = "HYBRID_V3_2_ALL_FAULT_LANDING_MEMORY_VER05"
LANDING_MEMORY_CACHE_ROOT = PART_B_RUN_ROOT / "_shared_feature_cache" / LANDING_MEMORY_CONTRACT
LANDING_MEMORY_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
TRAILING_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

LOOKBACK_DAYS = 30          # matches 11007a
HORIZON_DAYS = 14           # matches both notebooks
FEATURE_CUTOFF_SECONDS = 900  # matches 11002BC's existing landing-window cutoff
TOP_N_FEATURES = 500
TOP_N_FEATURES_EXPANDED = 1000        # same cap as before, but now RANKED, not arbitrary
ENGINE_WORKERS = min(50, os.cpu_count() or 8)
MIN_TS = pd.Timestamp("2000-01-01", tz="UTC")
MAX_TS = pd.Timestamp(datetime.now(timezone.utc))
STARTUP_TRANSIENT_CODES = {175, 176}
RANDOM_STATE = 11002
LANDING_MEMORY_LAGS = 5
LANDING_MEMORY_WORKERS = min(50, os.cpu_count() or 8)
LANDING_MEMORY_WEIGHTS = np.array([1.0, 0.80, 0.60, 0.40, 0.20], dtype=np.float32)
RECOVERY_CLEAN_LANDINGS = 3
LOOKBACK_NS = np.int64(pd.Timedelta(days=LOOKBACK_DAYS).value)

print({
    "part_b_run_root": str(PART_B_RUN_ROOT),
    "model_input_file": str(MODEL_INPUT_FILE),
    "v1_cache_root": str(TRAILING_CACHE_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "engine_workers": ENGINE_WORKERS,
    "fault_cache_root": str(FAULT_CACHE_ROOT),
    "landing_memory_cache_root": str(LANDING_MEMORY_CACHE_ROOT),
    "landing_memory_contract": LANDING_MEMORY_CONTRACT,
    "landing_memory_workers": LANDING_MEMORY_WORKERS,
    "top_n_features_expanded": TOP_N_FEATURES_EXPANDED,
})


{'part_b_run_root': '/raid3/e296408/All_ECFRs_working/ECFR_11002BC_ALL_ENGINE_REVISED_OPTIMIZED/data_richness/run_20260920_120031Z', 'model_input_file': '/raid3/e296408/All_ECFRs_working/ECFR_11002BC_ALL_ENGINE_REVISED_OPTIMIZED/data_richness/run_20260920_120031Z/data/landing_level_model_inputs.parquet', 'v1_cache_root': '/raid3/e296408/All_ECFRs_working/ECFR_11002BC_ALL_ENGINE_REVISED_OPTIMIZED/data_richness/run_20260920_120031Z/hybrid_trailing_history_benchmark/trailing_features_by_engine', 'output_root': '/raid3/e296408/All_ECFRs_working/ECFR_11002BC_ALL_ENGINE_REVISED_OPTIMIZED/data_richness/run_20260920_120031Z/hybrid_v3_2_all_fault_landing_memory_ver05', 'engine_workers': 50, 'fault_cache_root': '/raid3/e296408/All_ECFRs_working/ECFR_11002BC_ALL_ENGINE_REVISED_OPTIMIZED/data_richness/run_20260920_120031Z/hybrid_v3_fault_history_benchmark/fault_history_by_engine', 'landing_memory_cache_root': '/raid3/e296408/All_ECFRs_working/ECFR_11002BC_ALL_ENGINE_REVISED_OPTIMIZED/data_richness

## Trailing Window Features

The core new logic. For each landing, computes trailing 30-day aggregates
per status field using `searchsorted` + cumulative sums -- not a per-landing
scan, since a naive approach at over a million landings x hundreds of
fields would be far too slow.

**Tested against a brute-force per-window reference** before being trusted
here (realistic random histories, landings before/after all history, an
engine with zero events, an all-NaN field, and a performance check at
20,000 events x 2,000 landings -- 2.2ms).

In [2]:
# =============================================================================
# TRAILING WINDOW FEATURES (tested against a brute-force reference before
# being used here -- see test_trailing_window.py)
# =============================================================================
def trailing_window_features(landing_times_ns, event_times_ns, event_values, lookback_ns):
    """
    landing_times_ns: (M,) int64 ns, landing timestamps for one engine
    event_times_ns:   (N,) int64 ns, SORTED ascending, this engine's event times
    event_values:     (N,) float, this status field's value per event (NaN where absent)
    lookback_ns:      int64 ns window length

    Returns obs/active/last/changes arrays, one value per landing. Window is
    (landing - lookback, landing], matching 11007a's own window definition
    exactly, including its window-local "changes" semantics (the first
    non-null value observed within a window never counts as a change, even
    if it differs from a value just outside the window) -- confirmed via
    a brute-force per-window reference test, which caught an earlier,
    subtly different global-running version of this same function.
    """
    n = len(event_times_ns)
    if n == 0:
        m = len(landing_times_ns)
        zeros = np.zeros(m, dtype=np.int64)
        return {"obs": zeros, "active": zeros.copy(), "last": np.full(m, np.nan), "changes": zeros.copy()}

    notna = ~np.isnan(event_values)
    active = notna & (event_values != 0)

    cum_obs = np.concatenate([[0], np.cumsum(notna)])
    cum_active = np.concatenate([[0], np.cumsum(active)])

    last_nonnull_idx = np.maximum.accumulate(np.where(notna, np.arange(n), -1))

    next_nonnull_idx_raw = np.where(notna, np.arange(n), n)
    next_nonnull_idx_raw = np.minimum.accumulate(next_nonnull_idx_raw[::-1])[::-1]
    next_nonnull_idx = np.concatenate([next_nonnull_idx_raw, [n]])

    changed = np.zeros(n, dtype=bool)
    prev_nonnull_event_idx = np.full(n, -1, dtype=np.int64)
    seen_vals = event_values[notna]
    seen_idx = np.nonzero(notna)[0]
    if len(seen_vals) > 1:
        diffs = seen_vals[1:] != seen_vals[:-1]
        changed[seen_idx[1:]] = diffs
        prev_nonnull_event_idx[seen_idx[1:]] = seen_idx[:-1]
    cum_changes = np.concatenate([[0], np.cumsum(changed)])

    start_idx = np.searchsorted(event_times_ns, landing_times_ns - lookback_ns, side="right")
    end_idx = np.searchsorted(event_times_ns, landing_times_ns, side="right")

    obs = cum_obs[end_idx] - cum_obs[start_idx]
    act = cum_active[end_idx] - cum_active[start_idx]
    chg_raw = cum_changes[end_idx] - cum_changes[start_idx]

    first_nonnull_in_window = next_nonnull_idx[start_idx]
    has_first = first_nonnull_in_window < end_idx
    safe_first = np.clip(first_nonnull_in_window, 0, n - 1)
    boundary_crossing = (
        has_first
        & changed[safe_first]
        & (prev_nonnull_event_idx[safe_first] < start_idx)
    )
    chg = chg_raw - boundary_crossing.astype(np.int64)

    last_idx_in_window = np.where(end_idx > 0, last_nonnull_idx[np.maximum(end_idx - 1, 0)], -1)
    has_last = (end_idx > start_idx) & (last_idx_in_window >= start_idx)
    last_val = np.where(has_last, event_values[np.clip(last_idx_in_window, 0, n - 1)], np.nan)

    return {"obs": obs, "active": act, "last": last_val, "changes": chg}

## Phase 1: Load model_input, discover status fields

Loads the existing landing-level feature table and discovers every
decoded `sw1_`-`sw4_` status field across a sample of engines -- the same
discovery pattern `11007a` and `11002BC`'s own CELL 03 already use.

In [3]:
# =============================================================================
# PHASE 1: load model_input, discover status fields to build trailing
# history for (same discovery pattern as 11007a/11002BC's own CELL 03)
# =============================================================================
model_input = pd.read_parquet(MODEL_INPUT_FILE)
model_input["landing_datetime_utc"] = pd.to_datetime(model_input["landing_datetime_utc"], errors="coerce", utc=True)
model_input = model_input.loc[
    model_input["landing_datetime_utc"].notna()
    & model_input["landing_datetime_utc"].ge(MIN_TS)
    & model_input["landing_datetime_utc"].le(MAX_TS)
].reset_index(drop=True)

status_fields = set()
_sample_engine_dirs = list(ENGINE_ROOT.glob("engine=*"))[:50]
for d in _sample_engine_dirs:
    hist_file = d / "history_exact_deduplicated.parquet"
    if hist_file.is_file():
        status_fields.update(
            c for c in pq.ParquetFile(hist_file).schema_arrow.names if re.match(r"^sw[1-4]_", str(c), flags=re.I)
        )
status_fields = sorted(status_fields)
if not status_fields:
    raise RuntimeError("No sw1-4 status fields discovered -- check ENGINE_ROOT / a completed CELL 04 run")

print({"model_input_rows": len(model_input), "engines": model_input["engine_serial"].nunique(),
       "status_fields_discovered": len(status_fields)})

{'model_input_rows': 1669294, 'engines': 1916, 'status_fields_discovered': 79}


## Phase 2: Per-engine trailing-window computation (parallel)

Each worker loads **one engine's** raw history once and computes all
status fields' trailing features at all of that engine's landings in a
single pass, using the tested vectorized function above. Parallelized
across engines; restartable -- an engine whose cached output already
exists is skipped.

In [4]:
# =============================================================================
# PHASE 2: per-engine trailing-window feature computation (parallel across
# engines -- each worker loads its OWN engine's history once and computes
# all status fields' trailing features at all that engine's landings in one
# pass, using the tested vectorized function above)
# =============================================================================
def _compute_engine_trailing_features(args):
    engine, landing_ids, landing_times_ns, status_fields_list, engine_root_text, lookback_ns, cache_root_text = args
    started = time.time()
    result = {"engine_serial": engine, "status": "FAIL", "error": "", "rows": 0}
    out_path = Path(cache_root_text) / f"engine={engine}.parquet"

    try:
        hist_path = Path(engine_root_text) / f"engine={engine}" / "history_exact_deduplicated.parquet"
        if not hist_path.is_file():
            raise RuntimeError("No history file for this engine")

        schema_names = pq.ParquetFile(hist_path).schema_arrow.names
        available_fields = [f for f in status_fields_list if f in schema_names]
        if not available_fields or "record_datetime_utc" not in schema_names:
            out = pd.DataFrame({"engine_channel_landing_id": landing_ids})
            out.to_parquet(out_path, index=False, compression="zstd")
            result.update(status="PASS", rows=len(out), elapsed_seconds=round(time.time() - started, 3))
            return result

        hist = pd.read_parquet(hist_path, columns=["record_datetime_utc"] + available_fields)
        event_times = pd.to_datetime(hist["record_datetime_utc"], errors="coerce", utc=True)
        valid = event_times.notna()
        hist = hist.loc[valid].copy()
        event_times = event_times.loc[valid]
        # Converting to int64 while still a pandas Series (which correctly
        # handles the tz-aware -> epoch-ns conversion) before going to numpy
        # -- a plain .to_numpy() on a tz-aware Series can otherwise return an
        # object array of Timestamps depending on pandas version (caught by
        # testing).
        event_times_ns_unsorted = event_times.astype("int64").to_numpy()
        order = np.argsort(event_times_ns_unsorted)
        event_times_ns = event_times_ns_unsorted[order]
        hist = hist.iloc[order].reset_index(drop=True)

        out = {"engine_channel_landing_id": landing_ids}
        for field in available_fields:
            values = pd.to_numeric(hist[field], errors="coerce").to_numpy(dtype=float)
            feats = trailing_window_features(landing_times_ns, event_times_ns, values, lookback_ns)
            out[f"TRAIL30_{field}__OBS"] = feats["obs"]
            out[f"TRAIL30_{field}__ACTIVE"] = feats["active"]
            out[f"TRAIL30_{field}__LAST"] = feats["last"]
            out[f"TRAIL30_{field}__CHANGES"] = feats["changes"]

        out_df = pd.DataFrame(out)
        out_df.to_parquet(out_path, index=False, compression="zstd")
        result.update(status="PASS", rows=len(out_df), elapsed_seconds=round(time.time() - started, 3))

    except Exception:
        result["error"] = traceback.format_exc(limit=8)
        result["elapsed_seconds"] = round(time.time() - started, 3)
    return result


engines_to_process = sorted(model_input["engine_serial"].astype(str).unique())
tasks = []
for engine in engines_to_process:
    marker = TRAILING_CACHE_ROOT / f"engine={engine}.parquet"
    if marker.is_file():
        continue  # restartable: an engine already computed is skipped
    rows = model_input.loc[
        model_input["engine_serial"].astype(str) == engine, ["engine_channel_landing_id", "landing_datetime_utc"]
    ]
    # Passed directly as plain arrays, not round-tripped through JSON --
    # a to_json/read_json round-trip silently mangled the string ID
    # column in testing (pandas' JSON type inference is not reliable for
    # ID-like strings), and ProcessPoolExecutor already pickles numpy
    # arrays and DataFrames correctly across processes, so the JSON step
    # was both unnecessary and the actual source of a real bug.
    landing_ids = rows["engine_channel_landing_id"].astype(str).to_numpy()
    landing_times_ns = pd.to_datetime(rows["landing_datetime_utc"], utc=True).astype("int64").to_numpy()
    tasks.append((engine, landing_ids, landing_times_ns, status_fields, str(ENGINE_ROOT),
                  LOOKBACK_NS, str(TRAILING_CACHE_ROOT)))

print({"engines_total": len(engines_to_process), "already_cached": len(engines_to_process) - len(tasks),
       "remaining": len(tasks)})

trailing_results = []
if tasks:
    with ProcessPoolExecutor(max_workers=ENGINE_WORKERS) as executor:
        futures = {executor.submit(_compute_engine_trailing_features, t): t[0] for t in tasks}
        for i, future in enumerate(as_completed(futures), 1):
            engine = futures[future]
            try:
                r = future.result()
            except Exception:
                r = {"engine_serial": engine, "status": "FAIL", "error": traceback.format_exc(limit=8)}
            trailing_results.append(r)
            if i == 1 or i % 100 == 0 or i == len(tasks):
                print(f"[TRAIL {i:,}/{len(tasks):,}] {engine}: {r['status']}")

if trailing_results:
    failed = [r for r in trailing_results if r["status"] != "PASS"]
    if failed:
        pd.DataFrame(failed).to_csv(OUTPUT_ROOT / "trailing_feature_failures.csv", index=False)
        print(f"WARNING: {len(failed)} engine(s) failed trailing-feature computation -- see trailing_feature_failures.csv")

{'engines_total': 1916, 'already_cached': 1916, 'remaining': 0}


## Phase 3: Consolidate and join

Concatenates every engine's cached trailing-feature file and joins it
onto `model_input` by `engine_channel_landing_id`. The join key is forced
to a consistent string dtype on both sides -- defensive, since this is
exactly where a real bug surfaced during testing.

In [5]:
# =============================================================================
# PHASE 3: consolidate trailing features, join onto model_input
# =============================================================================
trailing_files = sorted(TRAILING_CACHE_ROOT.glob("engine=*.parquet"))
trailing_frames = [pd.read_parquet(f) for f in trailing_files]
trailing_all = pd.concat(trailing_frames, ignore_index=True, sort=False) if trailing_frames else pd.DataFrame()

# defensive: force a consistent string dtype for the join key on both
# sides -- the JSON round-trip through the worker processes can otherwise
# leave this column's dtype ambiguous (e.g. inferred as int64 if the IDs
# happen to look numeric), breaking the merge below with a dtype mismatch
model_input["engine_channel_landing_id"] = model_input["engine_channel_landing_id"].astype(str)
if not trailing_all.empty:
    trailing_all["engine_channel_landing_id"] = trailing_all["engine_channel_landing_id"].astype(str)

enriched = model_input.merge(trailing_all, on="engine_channel_landing_id", how="left")
trailing_cols = [c for c in enriched.columns if c.startswith("TRAIL30_")]
print({"trailing_feature_columns_added": len(trailing_cols), "enriched_rows": len(enriched)})

{'trailing_feature_columns_added': 316, 'enriched_rows': 1669294}


## Phase 3B: Cross-side trailing history and asymmetry

Pairs the focal engine with its resolved cross-side engine at the same aircraft landing event. For every focal trailing-history feature, it adds the partner value, signed focal-minus-partner delta, and absolute delta.


In [6]:
# =============================================================================
# PHASE 3B: exact-event cross-side trailing history + deltas
# =============================================================================
required_cross_side_columns = {
    "aircraft_landing_event_id", "engine_serial", "cross_side_engine_serial"
}
missing = required_cross_side_columns - set(enriched.columns)
if missing:
    raise RuntimeError(f"Cross-side experiment requires columns: {sorted(missing)}")

# Normalize IDs without converting missing values to the literal string "nan".
for c in ["aircraft_landing_event_id", "engine_serial", "cross_side_engine_serial"]:
    enriched[c] = enriched[c].astype("string").str.strip()

enriched["XSIDE_AVAILABLE"] = (
    enriched["aircraft_landing_event_id"].notna()
    & enriched["cross_side_engine_serial"].notna()
).astype("int8")

# One lookup record per aircraft event and engine. Channel duplicates should have
# identical trailing values because all values are computed as of the same landing.
lookup_columns = ["aircraft_landing_event_id", "engine_serial"] + trailing_cols
cross_lookup = enriched[lookup_columns].drop_duplicates(
    ["aircraft_landing_event_id", "engine_serial"], keep="first"
)
rename_map = {"engine_serial": "cross_side_engine_serial"}
rename_map.update({c: f"XSIDE_{c}" for c in trailing_cols})
cross_lookup = cross_lookup.rename(columns=rename_map)

before_rows = len(enriched)
enriched = enriched.merge(
    cross_lookup,
    on=["aircraft_landing_event_id", "cross_side_engine_serial"],
    how="left",
    validate="many_to_one",
)
if len(enriched) != before_rows:
    raise RuntimeError("Cross-side join changed row count")

xside_cols = [f"XSIDE_{c}" for c in trailing_cols]
# Use float32 blocks to control memory. Missing partner values remain NaN.
focal_block = enriched[trailing_cols].apply(pd.to_numeric, errors="coerce").astype("float32")
xside_block = enriched[xside_cols].apply(pd.to_numeric, errors="coerce").astype("float32")
xside_block.columns = trailing_cols
signed_delta = focal_block - xside_block
absolute_delta = signed_delta.abs()
signed_delta.columns = [f"DELTA_{c}" for c in trailing_cols]
absolute_delta.columns = [f"ABSDELTA_{c}" for c in trailing_cols]
enriched = pd.concat([enriched, signed_delta, absolute_delta], axis=1)

matched_rows = int(enriched[xside_cols].notna().any(axis=1).sum())
matched_events = int(enriched.loc[
    enriched[xside_cols].notna().any(axis=1), "aircraft_landing_event_id"
].nunique())

cross_side_audit = {
    "rows": len(enriched),
    "rows_with_declared_cross_side": int(enriched["XSIDE_AVAILABLE"].sum()),
    "rows_with_matched_cross_side_history": matched_rows,
    "aircraft_events_with_matched_cross_side_history": matched_events,
    "focal_trailing_columns": len(trailing_cols),
    "cross_side_columns": len(xside_cols),
    "signed_delta_columns": len(signed_delta.columns),
    "absolute_delta_columns": len(absolute_delta.columns),
}
(OUTPUT_ROOT / "cross_side_audit.json").write_text(
    json.dumps(cross_side_audit, indent=2), encoding="utf-8"
)
print(cross_side_audit)

del cross_lookup, focal_block, xside_block, signed_delta, absolute_delta


{'rows': 1669294, 'rows_with_declared_cross_side': 422863, 'rows_with_matched_cross_side_history': 396398, 'aircraft_events_with_matched_cross_side_history': 110221, 'focal_trailing_columns': 316, 'cross_side_columns': 316, 'signed_delta_columns': 316, 'absolute_delta_columns': 316}


## Phase 3C: Leakage-safe fault progression history

Computes focal-engine fault recency, frequency, diversity, persistence, and velocity at each landing cutoff. It uses each engine's exact deduplicated history, filters to the TR-family code set observed in the canonical fault artifact, and caches one Parquet file per engine. It then adds same-event cross-side fault values and deltas.


In [7]:
# =============================================================================
# PHASE 3C: focal + cross-side fault progression history
# =============================================================================
if not CANONICAL_FAULT_FILE.is_file():
    raise FileNotFoundError(CANONICAL_FAULT_FILE)
canonical_faults = pd.read_parquet(CANONICAL_FAULT_FILE, columns=["fault_code"])
TR_FAULT_CODES = set(pd.to_numeric(canonical_faults["fault_code"], errors="coerce").dropna().astype(int))
TR_FAULT_CODES -= STARTUP_TRANSIENT_CODES
if not TR_FAULT_CODES:
    raise RuntimeError("No TR-family fault codes were resolved from canonical_fault_events.parquet")

FAULT_LOOKBACK_DAYS = (7, 30, 90)
DAY_NS = np.int64(pd.Timedelta(days=1).value)

def fault_history_features(anchor_ns, event_ns, fault_codes, tr_code_set):
    """Return leakage-safe features using events <= each anchor."""
    m = len(anchor_ns)
    result = {
        "FAULTHIST_DAYS_SINCE_LAST_206": np.full(m, np.nan, dtype=np.float32),
        "FAULTHIST_DAYS_SINCE_LAST_TR": np.full(m, np.nan, dtype=np.float32),
    }
    for days in FAULT_LOOKBACK_DAYS:
        result[f"FAULTHIST_TR_COUNT_{days}D"] = np.zeros(m, dtype=np.int32)
        result[f"FAULTHIST_206_COUNT_{days}D"] = np.zeros(m, dtype=np.int32)
        result[f"FAULTHIST_DISTINCT_TR_{days}D"] = np.zeros(m, dtype=np.int16)
        result[f"FAULTHIST_MAX_REPEAT_TR_{days}D"] = np.zeros(m, dtype=np.int16)
    if len(event_ns) == 0:
        result["FAULTHIST_TR_RATE_7D"] = np.zeros(m, dtype=np.float32)
        result["FAULTHIST_TR_RATE_30D"] = np.zeros(m, dtype=np.float32)
        result["FAULTHIST_TR_ACCEL_7D_VS_30D"] = np.zeros(m, dtype=np.float32)
        result["FAULTHIST_206_ACCEL_7D_VS_30D"] = np.zeros(m, dtype=np.float32)
        return result

    order = np.argsort(event_ns)
    event_ns = event_ns[order]
    fault_codes = fault_codes[order]
    tr_mask = np.isin(fault_codes, list(tr_code_set))
    tr_ns, tr_codes = event_ns[tr_mask], fault_codes[tr_mask]
    f206_ns = event_ns[fault_codes == 206]

    end_tr = np.searchsorted(tr_ns, anchor_ns, side="right")
    end_206 = np.searchsorted(f206_ns, anchor_ns, side="right")
    has_tr = end_tr > 0
    has_206 = end_206 > 0
    result["FAULTHIST_DAYS_SINCE_LAST_TR"][has_tr] = (
        anchor_ns[has_tr] - tr_ns[end_tr[has_tr] - 1]
    ) / DAY_NS
    result["FAULTHIST_DAYS_SINCE_LAST_206"][has_206] = (
        anchor_ns[has_206] - f206_ns[end_206[has_206] - 1]
    ) / DAY_NS

    for days in FAULT_LOOKBACK_DAYS:
        width = np.int64(pd.Timedelta(days=days).value)
        start_tr = np.searchsorted(tr_ns, anchor_ns - width, side="right")
        start_206 = np.searchsorted(f206_ns, anchor_ns - width, side="right")
        result[f"FAULTHIST_TR_COUNT_{days}D"] = (end_tr - start_tr).astype(np.int32)
        result[f"FAULTHIST_206_COUNT_{days}D"] = (end_206 - start_206).astype(np.int32)
        distinct = result[f"FAULTHIST_DISTINCT_TR_{days}D"]
        max_repeat = result[f"FAULTHIST_MAX_REPEAT_TR_{days}D"]
        for i, (left, right) in enumerate(zip(start_tr, end_tr)):
            if right <= left:
                continue
            _, counts = np.unique(tr_codes[left:right], return_counts=True)
            distinct[i] = len(counts)
            max_repeat[i] = int(counts.max())

    result["FAULTHIST_TR_RATE_7D"] = result["FAULTHIST_TR_COUNT_7D"].astype(np.float32) / 7.0
    result["FAULTHIST_TR_RATE_30D"] = result["FAULTHIST_TR_COUNT_30D"].astype(np.float32) / 30.0
    result["FAULTHIST_TR_ACCEL_7D_VS_30D"] = (
        result["FAULTHIST_TR_RATE_7D"] / (result["FAULTHIST_TR_RATE_30D"] + np.float32(1e-3))
    )
    result["FAULTHIST_206_ACCEL_7D_VS_30D"] = (
        (result["FAULTHIST_206_COUNT_7D"].astype(np.float32) / 7.0)
        / ((result["FAULTHIST_206_COUNT_30D"].astype(np.float32) / 30.0) + np.float32(1e-3))
    )
    return result


def compute_engine_fault_history(args):
    engine, landing_ids, cutoff_ns, engine_root_text, cache_root_text, tr_codes = args
    out_path = Path(cache_root_text) / f"engine={engine}.parquet"
    result = {"engine_serial": engine, "status": "FAIL", "rows": 0, "error": ""}
    try:
        hist_path = Path(engine_root_text) / f"engine={engine}" / "history_exact_deduplicated.parquet"
        if not hist_path.is_file():
            raise FileNotFoundError(hist_path)
        names = set(pq.ParquetFile(hist_path).schema_arrow.names)
        required = {"record_datetime_utc", "fault_code"}
        if not required.issubset(names):
            frame = pd.DataFrame({"engine_channel_landing_id": landing_ids})
            for k, v in fault_history_features(cutoff_ns, np.array([], dtype=np.int64), np.array([], dtype=np.int64), tr_codes).items():
                frame[k] = v
        else:
            hist = pd.read_parquet(hist_path, columns=["record_datetime_utc", "fault_code"])
            ts = pd.to_datetime(hist["record_datetime_utc"], errors="coerce", utc=True)
            codes = pd.to_numeric(hist["fault_code"], errors="coerce")
            valid = ts.notna() & codes.notna()
            ev_ns = ts.loc[valid].astype("int64").to_numpy(dtype=np.int64)
            ev_codes = codes.loc[valid].astype(int).to_numpy(dtype=np.int64)
            # Exact event-level de-duplication within this feature family.
            if len(ev_ns):
                pairs = np.rec.fromarrays([ev_ns, ev_codes], names="t,c")
                _, keep = np.unique(pairs, return_index=True)
                ev_ns, ev_codes = ev_ns[keep], ev_codes[keep]
            frame = pd.DataFrame({"engine_channel_landing_id": landing_ids})
            for k, v in fault_history_features(cutoff_ns, ev_ns, ev_codes, tr_codes).items():
                frame[k] = v
        frame.to_parquet(out_path, index=False, compression="zstd")
        result.update(status="PASS", rows=len(frame))
    except Exception:
        result["error"] = traceback.format_exc(limit=8)
    return result

# The feature cutoff is landing + 900 s, matching the FF206 target boundary.
enriched["feature_cutoff_utc"] = enriched["landing_datetime_utc"] + pd.to_timedelta(FEATURE_CUTOFF_SECONDS, unit="s")
fault_tasks = []
for engine, rows in enriched.groupby(enriched["engine_serial"].astype(str), sort=False):
    marker = FAULT_CACHE_ROOT / f"engine={engine}.parquet"
    if marker.is_file():
        continue
    fault_tasks.append((
        str(engine),
        rows["engine_channel_landing_id"].astype(str).to_numpy(),
        rows["feature_cutoff_utc"].astype("int64").to_numpy(dtype=np.int64),
        str(ENGINE_ROOT), str(FAULT_CACHE_ROOT), TR_FAULT_CODES,
    ))
print({"fault_engines_total": enriched["engine_serial"].nunique(), "fault_engines_remaining": len(fault_tasks)})

fault_results = []
if fault_tasks:
    with ProcessPoolExecutor(max_workers=ENGINE_WORKERS) as executor:
        futures = {executor.submit(compute_engine_fault_history, t): t[0] for t in fault_tasks}
        for i, future in enumerate(as_completed(futures), 1):
            engine = futures[future]
            try:
                r = future.result()
            except Exception:
                r = {"engine_serial": engine, "status": "FAIL", "error": traceback.format_exc(limit=8)}
            fault_results.append(r)
            if i == 1 or i % 100 == 0 or i == len(fault_tasks):
                print(f"[FAULT {i:,}/{len(fault_tasks):,}] {engine}: {r['status']}")
if fault_results:
    failed = [r for r in fault_results if r["status"] != "PASS"]
    if failed:
        pd.DataFrame(failed).to_csv(OUTPUT_ROOT / "fault_history_failures.csv", index=False)
        raise RuntimeError(f"{len(failed)} engines failed fault-history generation")

fault_files = sorted(FAULT_CACHE_ROOT.glob("engine=*.parquet"))
fault_all = pd.concat([pd.read_parquet(p) for p in fault_files], ignore_index=True, sort=False)
fault_all["engine_channel_landing_id"] = fault_all["engine_channel_landing_id"].astype(str)
before_rows = len(enriched)
enriched["engine_channel_landing_id"] = enriched["engine_channel_landing_id"].astype(str)
enriched = enriched.merge(fault_all, on="engine_channel_landing_id", how="left", validate="one_to_one")
if len(enriched) != before_rows:
    raise RuntimeError("Fault-history join changed row count")
fault_cols = [c for c in enriched.columns if c.startswith("FAULTHIST_")]

# Cross-side fault history at the same aircraft event.
fault_lookup = enriched[["aircraft_landing_event_id", "engine_serial"] + fault_cols].drop_duplicates(
    ["aircraft_landing_event_id", "engine_serial"], keep="first"
).rename(columns={"engine_serial": "cross_side_engine_serial", **{c: f"XSIDE_{c}" for c in fault_cols}})
enriched = enriched.merge(
    fault_lookup,
    on=["aircraft_landing_event_id", "cross_side_engine_serial"],
    how="left", validate="many_to_one"
)
x_fault_cols = [f"XSIDE_{c}" for c in fault_cols]
focal_fault = enriched[fault_cols].apply(pd.to_numeric, errors="coerce").astype("float32")
xside_fault = enriched[x_fault_cols].apply(pd.to_numeric, errors="coerce").astype("float32")
xside_fault.columns = fault_cols
fault_delta = focal_fault - xside_fault
fault_absdelta = fault_delta.abs()
fault_delta.columns = [f"DELTA_{c}" for c in fault_cols]
fault_absdelta.columns = [f"ABSDELTA_{c}" for c in fault_cols]
enriched = pd.concat([enriched, fault_delta, fault_absdelta], axis=1)

fault_history_audit = {
    "tr_fault_code_count": len(TR_FAULT_CODES),
    "focal_fault_features": len(fault_cols),
    "cross_side_fault_features": len(x_fault_cols),
    "fault_delta_features": len(fault_delta.columns),
    "fault_absdelta_features": len(fault_absdelta.columns),
    "rows_with_any_focal_fault_history": int(enriched[fault_cols].notna().any(axis=1).sum()),
    "rows_with_any_cross_side_fault_history": int(enriched[x_fault_cols].notna().any(axis=1).sum()),
}
(OUTPUT_ROOT / "fault_history_audit.json").write_text(json.dumps(fault_history_audit, indent=2), encoding="utf-8")
print(fault_history_audit)
del fault_all, fault_lookup, focal_fault, xside_fault, fault_delta, fault_absdelta


{'fault_engines_total': 1916, 'fault_engines_remaining': 0}
{'tr_fault_code_count': 224, 'focal_fault_features': 18, 'cross_side_fault_features': 18, 'fault_delta_features': 18, 'fault_absdelta_features': 18, 'rows_with_any_focal_fault_history': 1669294, 'rows_with_any_cross_side_fault_history': 396398}


## Phase 3D: Detailed V3.1 memory plus compact all-code memory

The four detailed families retain nine features each. Every individual fault code in the resolved TR code set receives four compact focal features. Only weighted memory is mirrored cross-side, and only its absolute asymmetry is added, limiting width while testing every code.


In [8]:
# =============================================================================
# PHASE 3D: detailed family memory + compact all-code landing memory
# =============================================================================
if not CANONICAL_FAULT_FILE.is_file():
    raise FileNotFoundError(CANONICAL_FAULT_FILE)
canonical_faults = pd.read_parquet(CANONICAL_FAULT_FILE, columns=["fault_code"])
ALL_FAULT_CODES = sorted(set(pd.to_numeric(canonical_faults["fault_code"], errors="coerce").dropna().astype(int)) - STARTUP_TRANSIENT_CODES)
if not ALL_FAULT_CODES:
    raise RuntimeError("No usable fault codes resolved")
DETAILED_FAMILIES = {"FAULT204": {204}, "FAULT205": {205}, "FAULT206": {206}, "ANY_TR": set(TR_FAULT_CODES)}

# Separate versioned cache because VER05 V3.1 files do not contain LDGCODE columns.
ALL_CODE_CACHE_ROOT = PART_B_RUN_ROOT / "_shared_feature_cache" / "HYBRID_V3_2_ALL_FAULT_LANDING_MEMORY_VER05"
ALL_CODE_CACHE_ROOT.mkdir(parents=True, exist_ok=True)


def current_streak(history):
    value = 0
    for item in history[::-1]:
        if item: value += 1
        else: break
    return value


def landings_since_last(history):
    positions = np.flatnonzero(history)
    return int(len(history) - positions[-1]) if len(positions) else np.nan


def clean_streak_after_fault(history):
    positions = np.flatnonzero(history)
    return int(len(history) - 1 - positions[-1]) if len(positions) else np.nan


def detailed_memory(presence):
    n=len(presence); keys=["prev1","last3","last5","weighted","streak","since","clean","reappear","accel"]
    out={k:np.zeros(n,dtype=np.float32) for k in keys}; out["since"][:]=np.nan; out["clean"][:]=np.nan
    for i in range(n):
        history=presence[:i]; lag=history[-LANDING_MEMORY_LAGS:][::-1]; padded=np.zeros(LANDING_MEMORY_LAGS,dtype=np.float32); padded[:len(lag)]=lag
        out["prev1"][i]=padded[0]; out["last3"][i]=padded[:3].sum(); out["last5"][i]=padded.sum(); out["weighted"][i]=np.dot(padded,LANDING_MEMORY_WEIGHTS)
        out["streak"][i]=current_streak(history); out["since"][i]=landings_since_last(history); out["clean"][i]=clean_streak_after_fault(history)
        if len(history)>=RECOVERY_CLEAN_LANDINGS+2 and history[-1]==1:
            earlier=np.flatnonzero(history[:-1])
            if len(earlier): out["reappear"][i]=int(len(history)-1-earlier[-1]-1>=RECOVERY_CLEAN_LANDINGS)
        out["accel"][i]=padded[:2].mean()-padded[2:5].mean()
    return out


def compact_memory_matrix(bin_masks, n):
    """Returns prev1, last5, weighted, since for C fault-code rows x N landings."""
    c=bin_masks.shape[0]
    prev=np.zeros((c,n),dtype=np.float32); last5=np.zeros((c,n),dtype=np.float32); weighted=np.zeros((c,n),dtype=np.float32); since=np.full((c,n),np.nan,dtype=np.float32)
    # Shifted presence guarantees current bin exclusion.
    shifted=np.zeros_like(bin_masks,dtype=np.float32)
    if n>1: shifted[:,1:]=bin_masks[:,:-1]
    prev[:]=shifted
    for lag,weight in enumerate(LANDING_MEMORY_WEIGHTS, start=1):
        if lag<n:
            source=bin_masks[:,:n-lag]
            last5[:,lag:]+=source
            weighted[:,lag:]+=source*np.float32(weight)
    last_index=np.full(c,-1,dtype=np.int32)
    for i in range(n):
        since[:,i]=np.where(last_index>=0, i-last_index, np.nan)
        present=np.flatnonzero(bin_masks[:,i])
        if len(present): last_index[present]=i
    return prev,last5,weighted,since

# Synthetic semantics checks.
t=detailed_memory(np.array([0,1,1,0,0,0,1],dtype=np.int8))
assert t["prev1"][2]==1 and t["streak"][3]==2 and t["since"][6]==4 and t["clean"][6]==3
m=np.array([[1,0,0,0,0,0,0]],dtype=np.int8); p,l,w,ss=compact_memory_matrix(m,7)
assert p[0,0]==0 and p[0,1]==1 and np.isclose(w[0,5],.2) and np.isclose(w[0,6],0) and ss[0,1]==1
print("PASS: detailed and all-code compact-memory semantics")


def compute_engine_all_code_memory(args):
    engine, landing_ids, event_ids, cutoff_ns, engine_root_text, cache_root_text, all_codes = args
    path=Path(cache_root_text)/f"engine={engine}.parquet"; result={"engine_serial":engine,"status":"FAIL","rows":0,"error":""}
    try:
        order=np.lexsort((landing_ids.astype(str),cutoff_ns)); row_ids=landing_ids[order].astype(str); event_ids=event_ids[order].astype(str); cutoff_ns=cutoff_ns[order]
        unique=(pd.DataFrame({"event_id":event_ids,"cutoff_ns":cutoff_ns}).sort_values(["cutoff_ns","event_id"]).drop_duplicates("event_id").reset_index(drop=True))
        cutoffs=unique.cutoff_ns.to_numpy(dtype=np.int64); unique_ids=unique.event_id.astype(str).to_numpy(); n=len(cutoffs)
        event_ns=np.array([],dtype=np.int64); event_codes=np.array([],dtype=np.int64)
        hp=Path(engine_root_text)/f"engine={engine}"/"history_exact_deduplicated.parquet"
        if hp.is_file() and {"record_datetime_utc","fault_code"}.issubset(set(pq.ParquetFile(hp).schema_arrow.names)):
            h=pd.read_parquet(hp,columns=["record_datetime_utc","fault_code"]); ts=pd.to_datetime(h.record_datetime_utc,errors="coerce",utc=True); fc=pd.to_numeric(h.fault_code,errors="coerce"); valid=ts.notna()&fc.notna()
            event_ns=ts[valid].astype("int64").to_numpy(dtype=np.int64); event_codes=fc[valid].astype(int).to_numpy(dtype=np.int64)
            if len(event_ns):
                pair=np.rec.fromarrays([event_ns,event_codes],names="t,c");_,keep=np.unique(pair,return_index=True);event_ns,event_codes=event_ns[keep],event_codes[keep]
        bins=np.searchsorted(cutoffs,event_ns,side="left") if len(event_ns) else np.array([],dtype=np.int64); valid=bins<n; bins,event_codes=bins[valid],event_codes[valid]
        code_to_row={code:i for i,code in enumerate(all_codes)}; masks=np.zeros((len(all_codes),n),dtype=np.int8)
        for b,code in zip(bins,event_codes):
            r=code_to_row.get(int(code))
            if r is not None:masks[r,b]=1
        prev,last5,weighted,since=compact_memory_matrix(masks,n)
        data={"aircraft_landing_event_id":unique_ids}
        # Detailed known families.
        suffix={"prev1":"PREV1_PRESENT","last3":"LAST3_COUNT","last5":"LAST5_COUNT","weighted":"WEIGHTED_LAST5","streak":"PRIOR_STREAK","since":"LANDINGS_SINCE_LAST","clean":"CLEAN_STREAK_AFTER_LAST","reappear":"REAPPEARED_AFTER_CLEAN","accel":"RECENT_ACCELERATION"}
        for family,codes in DETAILED_FAMILIES.items():
            presence=np.zeros(n,dtype=np.int8); selected=bins[np.isin(event_codes,list(codes))]
            if len(selected):presence[np.unique(selected)]=1
            for key,values in detailed_memory(presence).items():data[f"LDGMEM_{family}_{suffix[key]}"]=values
        for r,code in enumerate(all_codes):
            token=f"{int(code):03d}"
            data[f"LDGCODE_{token}_PREV1_PRESENT"]=prev[r]
            data[f"LDGCODE_{token}_LAST5_COUNT"]=last5[r]
            data[f"LDGCODE_{token}_WEIGHTED_LAST5"]=weighted[r]
            data[f"LDGCODE_{token}_LANDINGS_SINCE_LAST"]=since[r]
        features=pd.DataFrame(data)
        output=pd.DataFrame({"engine_channel_landing_id":row_ids,"aircraft_landing_event_id":event_ids}).merge(features,on="aircraft_landing_event_id",how="left",validate="many_to_one").drop(columns="aircraft_landing_event_id")
        for c in output.columns:
            if c!="engine_channel_landing_id":output[c]=pd.to_numeric(output[c],errors="coerce").astype("float32")
        output.to_parquet(path,index=False,compression="zstd"); result.update(status="PASS",rows=len(output),columns=len(output.columns))
    except Exception:result["error"]=traceback.format_exc(limit=8)
    return result

memory_tasks=[]
for engine,rows in enriched.groupby(enriched.engine_serial.astype(str),sort=False):
    marker=ALL_CODE_CACHE_ROOT/f"engine={engine}.parquet"; valid=False
    if marker.is_file():
        try:
            pf=pq.ParquetFile(marker); names=set(pf.schema_arrow.names); valid=(pf.metadata.num_rows==len(rows) and "engine_channel_landing_id" in names and any(c.startswith("LDGCODE_") for c in names))
        except Exception:valid=False
    if valid:continue
    marker.unlink(missing_ok=True)
    memory_tasks.append((str(engine),rows.engine_channel_landing_id.astype(str).to_numpy(),rows.aircraft_landing_event_id.astype(str).to_numpy(),rows.feature_cutoff_utc.astype("int64").to_numpy(dtype=np.int64),str(ENGINE_ROOT),str(ALL_CODE_CACHE_ROOT),ALL_FAULT_CODES))
print({"engines":enriched.engine_serial.nunique(),"cached":enriched.engine_serial.nunique()-len(memory_tasks),"remaining":len(memory_tasks),"workers":LANDING_MEMORY_WORKERS,"fault_codes":len(ALL_FAULT_CODES)})
results=[]
if memory_tasks:
    with ProcessPoolExecutor(max_workers=LANDING_MEMORY_WORKERS) as executor:
        futures={executor.submit(compute_engine_all_code_memory,t):t[0] for t in memory_tasks}
        for i,future in enumerate(as_completed(futures),1):
            engine=futures[future]
            try:r=future.result()
            except Exception:r={"engine_serial":engine,"status":"FAIL","error":traceback.format_exc(limit=8)}
            results.append(r)
            if i==1 or i%100==0 or i==len(memory_tasks):print(f"[V3.2 {i:,}/{len(memory_tasks):,}] {engine}: {r['status']}")
failed=[r for r in results if r["status"]!="PASS"]
if failed:
    pd.DataFrame(failed).to_csv(OUTPUT_ROOT/"all_code_memory_failures_ver05.csv",index=False);raise RuntimeError(f"{len(failed)} engine failures")
files=sorted(ALL_CODE_CACHE_ROOT.glob("engine=*.parquet")); memory_all=pd.concat([pd.read_parquet(p) for p in files],ignore_index=True,sort=False);memory_all.engine_channel_landing_id=memory_all.engine_channel_landing_id.astype(str)
if memory_all.engine_channel_landing_id.duplicated().any():raise RuntimeError("duplicate landing IDs in V3.2 cache")
enriched.engine_channel_landing_id=enriched.engine_channel_landing_id.astype(str);before=len(enriched);enriched=enriched.merge(memory_all,on="engine_channel_landing_id",how="left",validate="one_to_one")
if len(enriched)!=before:raise RuntimeError("V3.2 join changed row count")
# Recreate the complete V3.1 detailed-family parent: focal, cross-side,
# signed delta, and absolute delta. This is required for a clean V3.1 replay.
detailed_cols=[c for c in enriched.columns if c.startswith("LDGMEM_")]
detailed_lookup=(enriched[["aircraft_landing_event_id","engine_serial"]+detailed_cols]
    .drop_duplicates(["aircraft_landing_event_id","engine_serial"])
    .rename(columns={"engine_serial":"cross_side_engine_serial",**{c:f"XSIDE_{c}" for c in detailed_cols}}))
enriched=enriched.merge(detailed_lookup,on=["aircraft_landing_event_id","cross_side_engine_serial"],how="left",validate="many_to_one")
xdetailed=[f"XSIDE_{c}" for c in detailed_cols]
focal_d=enriched[detailed_cols].astype("float32"); xside_d=enriched[xdetailed].astype("float32"); xside_d.columns=detailed_cols
delta_d=focal_d-xside_d; absdelta_d=delta_d.abs()
delta_d.columns=[f"DELTA_{c}" for c in detailed_cols]; absdelta_d.columns=[f"ABSDELTA_{c}" for c in detailed_cols]
enriched=pd.concat([enriched,delta_d,absdelta_d],axis=1)

# Cross-side only for weighted per-code memory, then absolute asymmetry only.
allcode_cols=[c for c in enriched.columns if c.startswith("LDGCODE_")]
weighted_cols=[c for c in allcode_cols if c.endswith("_WEIGHTED_LAST5")]
lookup=(enriched[["aircraft_landing_event_id","engine_serial"]+weighted_cols].drop_duplicates(["aircraft_landing_event_id","engine_serial"]).rename(columns={"engine_serial":"cross_side_engine_serial",**{c:f"XSIDE_{c}" for c in weighted_cols}}))
enriched=enriched.merge(lookup,on=["aircraft_landing_event_id","cross_side_engine_serial"],how="left",validate="many_to_one")
xweighted=[f"XSIDE_{c}" for c in weighted_cols]
focal=enriched[weighted_cols].astype("float32");xside=enriched[xweighted].astype("float32");xside.columns=weighted_cols
absdelta=(focal-xside).abs();absdelta.columns=[f"ABSDELTA_{c}" for c in weighted_cols];enriched=pd.concat([enriched,absdelta],axis=1)
# Support audit by code.
support=[]
for code in ALL_FAULT_CODES:
    token=f"{int(code):03d}"; prev=f"LDGCODE_{token}_PREV1_PRESENT"; last5=f"LDGCODE_{token}_LAST5_COUNT"
    support.append({"fault_code":int(code),"prev1_positive_rows":int((enriched[prev]>0).sum()),"last5_positive_rows":int((enriched[last5]>0).sum()),"eligible_support":bool((enriched[prev]>0).sum()>=100 or (enriched[last5]>0).sum()>=100)})
support_frame=pd.DataFrame(support)
support_frame.to_csv(OUTPUT_ROOT/"all_fault_code_support_ver05.csv",index=False)
ELIGIBLE_ALL_FAULT_CODES=set(support_frame.loc[support_frame["eligible_support"],"fault_code"].astype(int))
ELIGIBLE_ALL_CODE_TOKENS={f"{code:03d}" for code in ELIGIBLE_ALL_FAULT_CODES}
print({"resolved_fault_codes":len(ALL_FAULT_CODES),"eligible_fault_codes":len(ELIGIBLE_ALL_FAULT_CODES)})
all_code_audit={"contract":"HYBRID_V3_2_ALL_FAULT_LANDING_MEMORY_VER05","fault_codes":len(ALL_FAULT_CODES),"focal_all_code_features":len(allcode_cols),"cross_side_weighted_features":len(xweighted),"absolute_weighted_delta_features":len(absdelta.columns),"detailed_v31_focal_features":len(detailed_cols),"detailed_v31_cross_side_features":len(xdetailed),"detailed_v31_delta_features":len(delta_d.columns),"detailed_v31_absdelta_features":len(absdelta_d.columns),"workers":LANDING_MEMORY_WORKERS,"current_bin_excluded":True}
(OUTPUT_ROOT/"all_code_memory_audit_ver05.json").write_text(json.dumps(all_code_audit,indent=2),encoding="utf-8");print(all_code_audit)
del canonical_faults,memory_all,detailed_lookup,focal_d,xside_d,delta_d,absdelta_d,lookup,focal,xside,absdelta


PASS: detailed and all-code compact-memory semantics
{'engines': 1916, 'cached': 0, 'remaining': 1916, 'workers': 50, 'fault_codes': 224}
[V3.2 1/1,916] 118492: PASS
[V3.2 100/1,916] 130587: PASS
[V3.2 200/1,916] 131538: PASS
[V3.2 300/1,916] 131890: PASS
[V3.2 400/1,916] 136294: PASS
[V3.2 500/1,916] 136646: PASS
[V3.2 600/1,916] 136976: PASS
[V3.2 700/1,916] 144421: PASS
[V3.2 800/1,916] 145211: PASS
[V3.2 900/1,916] 145346: PASS
[V3.2 1,000/1,916] 147134: PASS
[V3.2 1,100/1,916] P118569: PASS
[V3.2 1,200/1,916] P130548: PASS
[V3.2 1,300/1,916] P131453: PASS
[V3.2 1,400/1,916] P136227: PASS
[V3.2 1,500/1,916] P136512: PASS
[V3.2 1,600/1,916] P136745: PASS
[V3.2 1,700/1,916] P144257: PASS
[V3.2 1,800/1,916] P144399: PASS
[V3.2 1,900/1,916] P145501: PASS
[V3.2 1,916/1,916] PI44305: PASS
{'resolved_fault_codes': 224, 'eligible_fault_codes': 169}
{'contract': 'HYBRID_V3_2_ALL_FAULT_LANDING_MEMORY_VER05', 'fault_codes': 224, 'focal_all_code_features': 896, 'cross_side_weighted_features': 

## Phase 4: Rebuild the FF206 target

Same explicit-FF-GMT label construction as the existing CELL 10, left
unchanged so the label definition stays directly comparable -- only the
feature set differs between this run and the baseline.

In [9]:
# =============================================================================
# PHASE 4: rebuild the FF206 target (same explicit-FF-GMT logic as the
# existing CELL 10 -- unchanged so the label definition stays comparable)
# =============================================================================
parts = sorted(ROW_V2_ROOT.glob("file_hash=*/part-*.parquet"))
selected_engine_set = set(enriched["engine_serial"].astype(str))
ff_frames = []
for index, path in enumerate(parts, 1):
    schema = set(pq.ParquetFile(path).schema_arrow.names)
    if not {"engine_serial", "section", "fault_code", "record_datetime_utc"}.issubset(schema):
        continue
    needed = [c for c in ["engine_serial", "section", "fault_code", "record_datetime_utc"] if c in schema]
    frame = pd.read_parquet(path, columns=needed)
    mask = (
        frame["engine_serial"].astype(str).isin(selected_engine_set)
        & frame["section"].fillna("").astype(str).str.upper().eq("FF_ATTRIBUTES")
        & pd.to_numeric(frame["fault_code"], errors="coerce").eq(206)
    )
    frame = frame.loc[mask].copy()
    if not frame.empty:
        frame["fault206_datetime_utc"] = pd.to_datetime(frame["record_datetime_utc"], errors="coerce", utc=True)
        ff_frames.append(frame)
    if index == 1 or index % 10000 == 0 or index == len(parts):
        print(f"Fault 206 scan {index:,}/{len(parts):,}; matching frames={len(ff_frames):,}")

if not ff_frames:
    raise RuntimeError("No corrected FF_ATTRIBUTES Fault 206 rows found")
fault206 = pd.concat(ff_frames, ignore_index=True, sort=False)
fault206 = fault206.loc[
    fault206["fault206_datetime_utc"].notna()
    & fault206["fault206_datetime_utc"].ge(MIN_TS)
    & fault206["fault206_datetime_utc"].le(MAX_TS)
].drop_duplicates(["engine_serial", "fault206_datetime_utc"])

fault_times_by_engine = {
    str(engine): np.sort(group["fault206_datetime_utc"].astype("int64").to_numpy(dtype=np.int64))
    for engine, group in fault206.groupby("engine_serial", sort=False)
}
cutoff_ns = enriched["feature_cutoff_utc"].astype("int64").to_numpy(dtype=np.int64)
horizon_ns = np.int64(pd.Timedelta(days=HORIZON_DAYS).value)
labels = np.zeros(len(enriched), dtype=np.int8)
for position, (engine, cutoff) in enumerate(zip(enriched["engine_serial"].astype(str), cutoff_ns)):
    times = fault_times_by_engine.get(engine)
    if times is None or len(times) == 0:
        continue
    idx = np.searchsorted(times, cutoff, side="right")
    if idx >= len(times):
        continue
    labels[position] = np.int8(np.int64(times[idx]) <= cutoff + horizon_ns)
TARGET = "FAULT206_WITHIN_14D_AFTER_WINDOW"
enriched[TARGET] = labels

Fault 206 scan 1/107,966; matching frames=0
Fault 206 scan 10,000/107,966; matching frames=400
Fault 206 scan 20,000/107,966; matching frames=860
Fault 206 scan 30,000/107,966; matching frames=1,314
Fault 206 scan 40,000/107,966; matching frames=1,735
Fault 206 scan 50,000/107,966; matching frames=2,132
Fault 206 scan 60,000/107,966; matching frames=2,521
Fault 206 scan 70,000/107,966; matching frames=2,944
Fault 206 scan 80,000/107,966; matching frames=3,370
Fault 206 scan 90,000/107,966; matching frames=3,802
Fault 206 scan 100,000/107,966; matching frames=4,218
Fault 206 scan 107,966/107,966; matching frames=4,578


## Phase 5: Isolated V3, V3.1, and V3.2 feature screens

V3 excludes all landing memory. V3.1 excludes all per-code compact memory. V3.2 includes every code. The V3.2 ranking is evaluated at both 500 and 1,000 selected features.


In [10]:
# =============================================================================
# PHASE 5: isolated V3 / V3.1 / V3.2 feature screens
# =============================================================================
identifier_columns={"aircraft_landing_event_id","engine_channel_landing_id","engine_landing_id","engine_serial","aircraft_identifier","aircraft_serial_number","aircraft_registration","cross_side_engine_serial","resolved_engine_model","ecu_channel","ecu_channel_norm","landing_datetime_utc","landing_ecu_time","feature_cutoff_utc",TARGET}
blocked_patterns=[r"FAULT__206_COUNT",r"TR_FAULT_206",r"(?<!\d)175(?!\d)",r"(?<!\d)176(?!\d)",r"STARTUP_175_176",r"FUTURE",r"TARGET",r"LABEL",r"NEXT_FAULT"]
def allowed_candidate(c):return c not in identifier_columns and not any(re.search(p,c,re.I) for p in blocked_patterns)
def is_detailed_memory(c):return "LDGMEM_" in c
def is_all_code_memory(c):return "LDGCODE_" in c
def all_code_supported(c):
    if "LDGCODE_" not in c:
        return True
    match=re.search(r"LDGCODE_(\d+)_",c)
    return bool(match and match.group(1) in ELIGIBLE_ALL_CODE_TOKENS)
all_candidates=[c for c in enriched.columns if allowed_candidate(c) and all_code_supported(c)]
v3_candidates=[c for c in all_candidates if not is_detailed_memory(c) and not is_all_code_memory(c)]
v31_candidates=[c for c in all_candidates if not is_all_code_memory(c)]
v32_candidates=list(all_candidates)
y_full=enriched[TARGET].astype(int)
def score_features(columns,tag):
    scored=[]
    for i,c in enumerate(columns,1):
        x=pd.to_numeric(enriched[c],errors="coerce");nonmissing=int(x.notna().sum())
        if nonmissing<100 or x.nunique(dropna=True)<=1:continue
        mask=x.notna()
        if y_full[mask].nunique()<2:continue
        corr=np.corrcoef(x[mask],y_full[mask])[0,1];scored.append((c,abs(corr) if np.isfinite(corr) else 0.0,nonmissing))
        if i==1 or i%1000==0 or i==len(columns):print(f"[{tag}] {i:,}/{len(columns):,}")
    scored.sort(key=lambda z:z[1],reverse=True)
    pd.DataFrame(scored,columns=["feature","abs_correlation_with_target","nonmissing_rows"]).to_csv(OUTPUT_ROOT/f"feature_ranking_{tag}.csv",index=False)
    return scored
v3_scored=score_features(v3_candidates,"hybrid_v3")
v31_scored=score_features(v31_candidates,"hybrid_v3_1")
v32_scored=score_features(v32_candidates,"hybrid_v3_2")
features_v3=[c for c,_,_ in v3_scored[:TOP_N_FEATURES]]
features_v31=[c for c,_,_ in v31_scored[:TOP_N_FEATURES]]
features_v32_500=[c for c,_,_ in v32_scored[:TOP_N_FEATURES]]
features_v32_1000=[c for c,_,_ in v32_scored[:TOP_N_FEATURES_EXPANDED]]
if not all([features_v3,features_v31,features_v32_500,features_v32_1000]):raise RuntimeError("No usable features")
selection_audit={"v3":len(features_v3),"v31":len(features_v31),"v32_500":len(features_v32_500),"v32_1000":len(features_v32_1000),"v31_memory_selected":sum(is_detailed_memory(c) for c in features_v31),"v32_500_all_code_selected":sum(is_all_code_memory(c) for c in features_v32_500),"v32_1000_all_code_selected":sum(is_all_code_memory(c) for c in features_v32_1000)}
(OUTPUT_ROOT/"feature_selection_audit_ver05.json").write_text(json.dumps(selection_audit,indent=2),encoding="utf-8");print(selection_audit)


[hybrid_v3] 1/8,159
[hybrid_v3] 1,000/8,159
[hybrid_v3] 2,000/8,159
[hybrid_v3] 3,000/8,159
[hybrid_v3] 4,000/8,159
[hybrid_v3] 5,000/8,159
[hybrid_v3] 6,000/8,159
[hybrid_v3] 7,000/8,159
[hybrid_v3] 8,000/8,159
[hybrid_v3] 8,159/8,159
[hybrid_v3_1] 1/8,303
[hybrid_v3_1] 1,000/8,303
[hybrid_v3_1] 2,000/8,303
[hybrid_v3_1] 3,000/8,303
[hybrid_v3_1] 4,000/8,303
[hybrid_v3_1] 5,000/8,303
[hybrid_v3_1] 6,000/8,303
[hybrid_v3_1] 7,000/8,303
[hybrid_v3_1] 8,000/8,303
[hybrid_v3_1] 8,303/8,303
[hybrid_v3_2] 1/9,317
[hybrid_v3_2] 1,000/9,317
[hybrid_v3_2] 2,000/9,317
[hybrid_v3_2] 3,000/9,317
[hybrid_v3_2] 4,000/9,317
[hybrid_v3_2] 5,000/9,317
[hybrid_v3_2] 6,000/9,317
[hybrid_v3_2] 7,000/9,317
[hybrid_v3_2] 8,000/9,317
[hybrid_v3_2] 9,000/9,317
[hybrid_v3_2] 9,317/9,317
{'v3': 500, 'v31': 500, 'v32_500': 500, 'v32_1000': 1000, 'v31_memory_selected': 18, 'v32_500_all_code_selected': 92, 'v32_1000_all_code_selected': 107}


## Phase 6: Matched V3, V3.1, V3.2-500, and V3.2-1000 evaluation

All variants use identical rows, target, aircraft groups, folds, model hyperparameters, and ranking method. Only candidate family and feature cap differ.


In [ ]:
# =============================================================================
# PHASE 6:
# MATCHED V3 / V3.1 / V3.2-500 / V3.2-1000 GROUPED EVALUATION
#
# Fixes:
#   1. Converts all NumPy and pandas values before JSON serialization.
#   2. Explicitly converts promotion-gate results to native Python bool.
#   3. Verifies the written JSON by reading it back.
#   4. Preserves matched rows, groups, folds, learner, and hyperparameters.
# =============================================================================

from pathlib import Path

from sklearn.ensemble import (
    HistGradientBoostingClassifier,
)
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GroupKFold,
)
from sklearn.utils.class_weight import (
    compute_sample_weight,
)


# -----------------------------------------------------------------------------
# JSON serialization helper
# -----------------------------------------------------------------------------

def json_safe(value):
    """
    Recursively convert NumPy, pandas, Path, tuple, and set values
    into native Python values supported by json.dumps().
    """
    if isinstance(value, dict):
        return {
            str(key): json_safe(item)
            for key, item in value.items()
        }

    if isinstance(
        value,
        (list, tuple, set),
    ):
        return [
            json_safe(item)
            for item in value
        ]

    if isinstance(value, np.ndarray):
        return json_safe(
            value.tolist()
        )

    if isinstance(value, np.bool_):
        return bool(value)

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return float(value)

    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    if isinstance(value, Path):
        return str(value)

    if value is pd.NA:
        return None

    return value


# -----------------------------------------------------------------------------
# Grouping and target
# -----------------------------------------------------------------------------

if (
    "aircraft_identifier"
    in enriched.columns
    and enriched[
        "aircraft_identifier"
    ].notna().sum() > 0
):
    groups = (
        enriched[
            "aircraft_identifier"
        ]
        .astype("string")
        .fillna(
            enriched[
                "engine_serial"
            ].astype(str)
        )
    )
else:
    groups = (
        enriched[
            "engine_serial"
        ].astype(str)
    )

y = (
    enriched[TARGET]
    .astype(int)
)

group_target_frame = pd.DataFrame(
    {
        "group": groups,
        "target": y,
    }
)

positive_groups = int(
    group_target_frame
    .groupby("group")["target"]
    .max()
    .sum()
)

n_splits = min(
    5,
    positive_groups,
)

if n_splits < 2:
    raise RuntimeError(
        "Grouped validation requires at least "
        "two positive groups"
    )

group_kfold = GroupKFold(
    n_splits=n_splits,
)

splits = list(
    group_kfold.split(
        enriched,
        y,
        groups,
    )
)

print(
    {
        "evaluation_rows":
            len(enriched),
        "positive_rows":
            int(y.sum()),
        "prevalence":
            float(y.mean()),
        "groups":
            int(groups.nunique()),
        "positive_groups":
            positive_groups,
        "folds":
            n_splits,
    }
)


# -----------------------------------------------------------------------------
# Matched model evaluation
# -----------------------------------------------------------------------------

def evaluate(
    feature_list,
    model_tag,
):
    """
    Evaluate one feature variant using the shared grouped folds.

    Feature selection has already occurred in Phase 5. Within each fold,
    unusable constant or entirely missing features are removed using
    training data only.
    """
    if not feature_list:
        raise RuntimeError(
            f"{model_tag} received an empty feature list"
        )

    X = (
        enriched[feature_list]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
        .astype("float32")
    )

    oof_probability = np.full(
        len(enriched),
        np.nan,
        dtype=float,
    )

    fold_rows = []

    for fold, (
        train_idx,
        valid_idx,
    ) in enumerate(
        splits,
        start=1,
    ):
        X_train = X.iloc[
            train_idx
        ]

        X_valid = X.iloc[
            valid_idx
        ]

        y_train = y.iloc[
            train_idx
        ]

        y_valid = y.iloc[
            valid_idx
        ]

        fold_features = [
            feature
            for feature
            in feature_list
            if (
                X_train[
                    feature
                ].notna().sum()
                > 0
                and
                X_train[
                    feature
                ].nunique(
                    dropna=True
                )
                > 1
            )
        ]

        if not fold_features:
            raise RuntimeError(
                f"{model_tag}, fold {fold}: "
                "no usable training features"
            )

        if y_train.nunique() < 2:
            raise RuntimeError(
                f"{model_tag}, fold {fold}: "
                "training target has fewer than two classes"
            )

        if y_valid.nunique() < 2:
            raise RuntimeError(
                f"{model_tag}, fold {fold}: "
                "validation target has fewer than two classes"
            )

        model = (
            HistGradientBoostingClassifier(
                learning_rate=0.07,
                max_iter=100,
                max_leaf_nodes=31,
                min_samples_leaf=50,
                l2_regularization=1.0,
                early_stopping=True,
                validation_fraction=0.10,
                n_iter_no_change=10,
                random_state=(
                    RANDOM_STATE
                    + fold
                ),
            )
        )

        sample_weights = (
            compute_sample_weight(
                class_weight="balanced",
                y=y_train,
            )
        )

        model.fit(
            X_train[
                fold_features
            ],
            y_train,
            sample_weight=(
                sample_weights
            ),
        )

        probability = (
            model.predict_proba(
                X_valid[
                    fold_features
                ]
            )[:, 1]
        )

        oof_probability[
            valid_idx
        ] = probability

        fold_roc_auc = float(
            roc_auc_score(
                y_valid,
                probability,
            )
        )

        fold_pr_auc = float(
            average_precision_score(
                y_valid,
                probability,
            )
        )

        fold_prevalence = float(
            y_valid.mean()
        )

        fold_pr_lift = float(
            fold_pr_auc
            / fold_prevalence
        )

        fold_rows.append(
            {
                "model":
                    str(model_tag),
                "fold":
                    int(fold),
                "train_rows":
                    int(len(train_idx)),
                "valid_rows":
                    int(len(valid_idx)),
                "valid_positive_rows":
                    int(y_valid.sum()),
                "features_requested":
                    int(len(feature_list)),
                "features_used":
                    int(len(fold_features)),
                "detailed_memory_features_used":
                    int(
                        sum(
                            "LDGMEM_"
                            in feature
                            for feature
                            in fold_features
                        )
                    ),
                "all_code_memory_features_used":
                    int(
                        sum(
                            "LDGCODE_"
                            in feature
                            for feature
                            in fold_features
                        )
                    ),
                "roc_auc":
                    fold_roc_auc,
                "pr_auc":
                    fold_pr_auc,
                "prevalence":
                    fold_prevalence,
                "pr_lift":
                    fold_pr_lift,
            }
        )

        print(
            {
                "model":
                    model_tag,
                "fold":
                    fold,
                "features_used":
                    len(fold_features),
                "roc_auc":
                    round(
                        fold_roc_auc,
                        6,
                    ),
                "pr_auc":
                    round(
                        fold_pr_auc,
                        6,
                    ),
                "pr_lift":
                    round(
                        fold_pr_lift,
                        6,
                    ),
            }
        )

    valid_oof = np.isfinite(
        oof_probability
    )

    if not valid_oof.all():
        missing_oof = int(
            (~valid_oof).sum()
        )

        raise RuntimeError(
            f"{model_tag}: "
            f"{missing_oof:,} rows lack OOF predictions"
        )

    overall_roc_auc = float(
        roc_auc_score(
            y[valid_oof],
            oof_probability[
                valid_oof
            ],
        )
    )

    overall_pr_auc = float(
        average_precision_score(
            y[valid_oof],
            oof_probability[
                valid_oof
            ],
        )
    )

    overall_prevalence = float(
        y[valid_oof].mean()
    )

    overall_pr_lift = float(
        overall_pr_auc
        / overall_prevalence
    )

    summary = {
        "model":
            str(model_tag),
        "rows":
            int(valid_oof.sum()),
        "positive_rows":
            int(
                y[
                    valid_oof
                ].sum()
            ),
        "positive_engines":
            int(
                enriched.loc[
                    valid_oof
                    & y.eq(1),
                    "engine_serial",
                ].nunique()
            ),
        "positive_groups":
            int(positive_groups),
        "features_used":
            int(len(feature_list)),
        "detailed_memory_features_selected":
            int(
                sum(
                    "LDGMEM_"
                    in feature
                    for feature
                    in feature_list
                )
            ),
        "all_code_memory_features_selected":
            int(
                sum(
                    "LDGCODE_"
                    in feature
                    for feature
                    in feature_list
                )
            ),
        "roc_auc":
            overall_roc_auc,
        "pr_auc":
            overall_pr_auc,
        "prevalence":
            overall_prevalence,
        "pr_lift":
            overall_pr_lift,
    }

    return (
        summary,
        pd.DataFrame(
            fold_rows
        ),
        oof_probability,
    )


# -----------------------------------------------------------------------------
# Run matched variants
# -----------------------------------------------------------------------------

summary_v3, folds_v3, oof_v3 = (
    evaluate(
        features_v3,
        "HYBRID_V3",
    )
)

summary_v31, folds_v31, oof_v31 = (
    evaluate(
        features_v31,
        "HYBRID_V3_1",
    )
)

summary_v32_500, folds_v32_500, oof_v32_500 = (
    evaluate(
        features_v32_500,
        "HYBRID_V3_2_TOP500",
    )
)

summary_v32_1000, folds_v32_1000, oof_v32_1000 = (
    evaluate(
        features_v32_1000,
        "HYBRID_V3_2_TOP1000",
    )
)


# -----------------------------------------------------------------------------
# Save fold-level results
# -----------------------------------------------------------------------------

fold_results = pd.concat(
    [
        folds_v3,
        folds_v31,
        folds_v32_500,
        folds_v32_1000,
    ],
    ignore_index=True,
)

fold_results_path = (
    OUTPUT_ROOT
    / "fold_results_v3_v31_v32_ver05.csv"
)

fold_results.to_csv(
    fold_results_path,
    index=False,
)


# -----------------------------------------------------------------------------
# Save OOF predictions for later analysis
# -----------------------------------------------------------------------------

oof_results = pd.DataFrame(
    {
        "engine_channel_landing_id":
            enriched[
                "engine_channel_landing_id"
            ].astype(str),
        "engine_serial":
            enriched[
                "engine_serial"
            ].astype(str),
        "target":
            y.astype(np.int8),
        "oof_v3":
            oof_v3,
        "oof_v31":
            oof_v31,
        "oof_v32_top500":
            oof_v32_500,
        "oof_v32_top1000":
            oof_v32_1000,
    }
)

oof_results_path = (
    OUTPUT_ROOT
    / "oof_predictions_v3_v31_v32_ver05.parquet"
)

oof_results.to_parquet(
    oof_results_path,
    index=False,
    compression="zstd",
)


# -----------------------------------------------------------------------------
# Select best parent and best V3.2 variant by PR AUC
# -----------------------------------------------------------------------------

parent_summaries = [
    summary_v3,
    summary_v31,
]

v32_summaries = [
    summary_v32_500,
    summary_v32_1000,
]

best_parent = max(
    parent_summaries,
    key=lambda result: float(
        result["pr_auc"]
    ),
)

best_v32 = max(
    v32_summaries,
    key=lambda result: float(
        result["pr_auc"]
    ),
)


# -----------------------------------------------------------------------------
# Incremental metrics
# -----------------------------------------------------------------------------

delta = {
    "roc_auc": float(
        best_v32["roc_auc"]
        - best_parent["roc_auc"]
    ),
    "pr_auc": float(
        best_v32["pr_auc"]
        - best_parent["pr_auc"]
    ),
    "pr_lift": float(
        best_v32["pr_lift"]
        - best_parent["pr_lift"]
    ),
    "relative_pr_lift_pct": float(
        100.0
        * (
            best_v32["pr_lift"]
            / best_parent["pr_lift"]
            - 1.0
        )
    ),
}


# -----------------------------------------------------------------------------
# Promotion gate
#
# Explicit bool() conversions are required because NumPy comparison results
# are not supported by Python's standard JSON encoder.
# -----------------------------------------------------------------------------

promotion_gate = {
    "roc_gain_at_least_0_01": bool(
        delta["roc_auc"]
        >= 0.01
    ),
    "relative_pr_lift_gain_at_least_5pct": bool(
        delta[
            "relative_pr_lift_pct"
        ]
        >= 5.0
    ),
}

promotion_gate[
    "exploratory_gate_pass"
] = bool(
    any(
        promotion_gate.values()
    )
)


# -----------------------------------------------------------------------------
# Final comparison object
# -----------------------------------------------------------------------------

comparison = {
    "hybrid_v3":
        summary_v3,
    "hybrid_v3_1":
        summary_v31,
    "hybrid_v3_2_top500":
        summary_v32_500,
    "hybrid_v3_2_top1000":
        summary_v32_1000,
    "best_parent_by_pr_auc":
        str(
            best_parent[
                "model"
            ]
        ),
    "best_v32_by_pr_auc":
        str(
            best_v32[
                "model"
            ]
        ),
    "delta_best_v32_minus_best_parent":
        delta,
    "promotion_gate":
        promotion_gate,
    "all_code_audit":
        all_code_audit,
    "artifacts": {
        "fold_results":
            str(
                fold_results_path
            ),
        "oof_predictions":
            str(
                oof_results_path
            ),
    },
    "method_note": (
        "Exploratory global feature screening; "
        "repeat with fold-local feature selection "
        "or a locked out-of-time holdout before promotion."
    ),
}


# -----------------------------------------------------------------------------
# Convert complete nested object to JSON-safe native types
# -----------------------------------------------------------------------------

comparison_safe = json_safe(
    comparison
)

summary_text = json.dumps(
    comparison_safe,
    indent=2,
    allow_nan=True,
)


# -----------------------------------------------------------------------------
# Write and verify JSON result
# -----------------------------------------------------------------------------

summary_path = (
    OUTPUT_ROOT
    / "run_summary_v3_2_ver05.json"
)

summary_path.write_text(
    summary_text,
    encoding="utf-8",
)

with summary_path.open(
    "r",
    encoding="utf-8",
) as handle:
    verified_summary = json.load(
        handle
    )

assert isinstance(
    verified_summary[
        "promotion_gate"
    ][
        "roc_gain_at_least_0_01"
    ],
    bool,
)

assert isinstance(
    verified_summary[
        "promotion_gate"
    ][
        "relative_pr_lift_gain_at_least_5pct"
    ],
    bool,
)

assert isinstance(
    verified_summary[
        "promotion_gate"
    ][
        "exploratory_gate_pass"
    ],
    bool,
)

assert (
    verified_summary[
        "best_v32_by_pr_auc"
    ]
    == str(
        best_v32[
            "model"
        ]
    )
)


# -----------------------------------------------------------------------------
# Final output
# -----------------------------------------------------------------------------

print(
    "\n"
    + "=" * 78
)

print(
    "HYBRID V3 / V3.1 / V3.2 RESULTS"
)

print(
    "=" * 78
)

print(
    summary_text
)

print(
    {
        "summary_file":
            str(summary_path),
        "summary_verified":
            summary_path.is_file(),
        "fold_results_file":
            str(
                fold_results_path
            ),
        "oof_predictions_file":
            str(
                oof_results_path
            ),
        "best_parent":
            verified_summary[
                "best_parent_by_pr_auc"
            ],
        "best_v32":
            verified_summary[
                "best_v32_by_pr_auc"
            ],
        "promotion_gate":
            verified_summary[
                "promotion_gate"
            ],
    }
)

print(
    "PASS: Hybrid V3.2 all-fault "
    "landing-memory benchmark completed"
)

TypeError: Object of type bool is not JSON serializable

In [14]:
# =============================================================================
# V3.2 FINAL SUMMARY RECOVERY
# Fixes NumPy / pandas JSON serialization without rerunning the models
# =============================================================================

def json_safe(value):
    """
    Recursively convert NumPy and pandas objects into native Python
    values accepted by json.dumps().
    """
    if isinstance(value, dict):
        return {
            str(key): json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple, set)):
        return [
            json_safe(item)
            for item in value
        ]

    if isinstance(value, np.ndarray):
        return json_safe(
            value.tolist()
        )

    if isinstance(value, np.bool_):
        return bool(value)

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return float(value)

    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    if isinstance(value, Path):
        return str(value)

    if value is pd.NA:
        return None

    return value


# Recalculate the comparison objects using explicit native Python types.
best_parent = max(
    [
        summary_v3,
        summary_v31,
    ],
    key=lambda result: float(
        result["pr_auc"]
    ),
)

best_v32 = max(
    [
        summary_v32_500,
        summary_v32_1000,
    ],
    key=lambda result: float(
        result["pr_auc"]
    ),
)

delta = {
    "roc_auc": float(
        best_v32["roc_auc"]
        - best_parent["roc_auc"]
    ),
    "pr_auc": float(
        best_v32["pr_auc"]
        - best_parent["pr_auc"]
    ),
    "relative_pr_lift_pct": float(
        100.0
        * (
            best_v32["pr_lift"]
            / best_parent["pr_lift"]
            - 1.0
        )
    ),
}

promotion_gate = {
    "roc_gain_at_least_0_01": bool(
        delta["roc_auc"] >= 0.01
    ),
    "relative_pr_lift_gain_at_least_5pct": bool(
        delta["relative_pr_lift_pct"] >= 5.0
    ),
}

promotion_gate[
    "exploratory_gate_pass"
] = bool(
    any(promotion_gate.values())
)

comparison = {
    "hybrid_v3": summary_v3,
    "hybrid_v3_1": summary_v31,
    "hybrid_v3_2_top500":
        summary_v32_500,
    "hybrid_v3_2_top1000":
        summary_v32_1000,
    "best_parent_by_pr_auc": str(
        best_parent["model"]
    ),
    "best_v32_by_pr_auc": str(
        best_v32["model"]
    ),
    "delta_best_v32_minus_best_parent":
        delta,
    "promotion_gate":
        promotion_gate,
    "all_code_audit":
        all_code_audit,
    "method_note": (
        "Exploratory global feature screening; "
        "repeat fold-local or out-of-time before promotion."
    ),
}

comparison_safe = json_safe(
    comparison
)

summary_text = json.dumps(
    comparison_safe,
    indent=2,
    allow_nan=True,
)

summary_path = (
    OUTPUT_ROOT
    / "run_summary_v3_2_ver05.json"
)

summary_path.write_text(
    summary_text,
    encoding="utf-8",
)

# Verification: read the written JSON back immediately.
with summary_path.open(
    "r",
    encoding="utf-8",
) as handle:
    verified_summary = json.load(
        handle
    )

assert (
    verified_summary[
        "best_v32_by_pr_auc"
    ]
    == str(best_v32["model"])
)

assert isinstance(
    verified_summary[
        "promotion_gate"
    ][
        "exploratory_gate_pass"
    ],
    bool,
)

print(summary_text)

print(
    {
        "summary_file":
            str(summary_path),
        "summary_file_exists":
            summary_path.is_file(),
        "best_parent":
            verified_summary[
                "best_parent_by_pr_auc"
            ],
        "best_v32":
            verified_summary[
                "best_v32_by_pr_auc"
            ],
        "promotion_gate":
            verified_summary[
                "promotion_gate"
            ],
    }
)

print(
    "PASS: Hybrid V3.2 all-fault landing-memory "
    "summary written and verified"
)

{
  "hybrid_v3": {
    "model": "HYBRID_V3",
    "rows": 1669294,
    "positive_rows": 87715,
    "features_used": 500,
    "detailed_memory_selected": 0,
    "all_code_memory_selected": 0,
    "roc_auc": 0.7388961465858501,
    "pr_auc": 0.1162603846489781,
    "prevalence": 0.05254616622356517,
    "pr_lift": 2.21253790722489
  },
  "hybrid_v3_1": {
    "model": "HYBRID_V3_1",
    "rows": 1669294,
    "positive_rows": 87715,
    "features_used": 500,
    "detailed_memory_selected": 18,
    "all_code_memory_selected": 0,
    "roc_auc": 0.7414884891258254,
    "pr_auc": 0.11539950412611284,
    "prevalence": 0.05254616622356517,
    "pr_lift": 2.1961545897588257
  },
  "hybrid_v3_2_top500": {
    "model": "HYBRID_V3_2_TOP500",
    "rows": 1669294,
    "positive_rows": 87715,
    "features_used": 500,
    "detailed_memory_selected": 18,
    "all_code_memory_selected": 92,
    "roc_auc": 0.7398184771298144,
    "pr_auc": 0.12159408609379757,
    "prevalence": 0.05254616622356517,
    "pr